In [4]:
import pandas as pd
import numpy as np

In [5]:
main_race_res_2019 = pd.read_csv('main_race_result(processed)/formula1_2019season_raceResults.csv')
main_race_res_2020 = pd.read_csv('main_race_result(processed)/formula1_2020season_raceResults.csv')
main_race_res_2021 = pd.read_csv('main_race_result(processed)/formula1_2021season_raceResults.csv')
main_race_res_2022 = pd.read_csv('main_race_result(processed)/formula1_2022season_raceResults.csv')
main_race_res_2023 = pd.read_csv('main_race_result(processed)/formula1_2023season_raceResults.csv')
main_race_res_2024 = pd.read_csv('main_race_result(processed)/formula1_2024season_raceResults.csv')
main_race_res_2025 = pd.read_csv('main_race_result(processed)/formula1_2025season_raceResults.csv')

sprint_quali_2023 = pd.read_csv('quali_both_result(processed)/sprint_quali_2023.csv')
sprint_quali_2024 = pd.read_csv('quali_both_result(processed)/sprint_quali_2024.csv')
sprint_quali_2025 = pd.read_csv('quali_both_result(processed)/sprint_quali_2025.csv')

sprint_race_res_2021 = pd.read_csv('sprint_race_result(processed)/sprint_2021_result.csv')
sprint_race_res_2022 = pd.read_csv('sprint_race_result(processed)/sprint_2022_result.csv')
sprint_race_res_2023 = pd.read_csv('sprint_race_result(processed)/sprint_2023_result.csv')
sprint_race_res_2024 = pd.read_csv('sprint_race_result(processed)/sprint_2024_result.csv')
sprint_race_res_2025 = pd.read_csv('sprint_race_result(processed)/sprint_2025_result.csv')

In [73]:
def load_quali(year):
    df = pd.read_csv(f'quali_both_result(processed)/race_quali_{year}.csv')
    
    # Normalize position column name (2022+ uses 'Position' with capital P)
    if 'Position' in df.columns:
        df = df.rename(columns={'Position': 'position'})
    
    # Only keep what's needed — prevents any column collision during merge
    keep = ['raceId', 'kaggle_driver_id', 'position', 
            'participated_q2', 'participated_q3', 
            'q1_time_sec', 'q2_time_sec', 'q3_time_sec','Track']
    df = df[keep]
    
    return df

main_race_quali_2019 = load_quali(2019)
main_race_quali_2020 = load_quali(2020)
main_race_quali_2021 = load_quali(2021)
main_race_quali_2022 = load_quali(2022)
main_race_quali_2023 = load_quali(2023)
main_race_quali_2024 = load_quali(2024)

In [7]:
print("Race rows:", len(main_race_res_2019))
print("Quali rows:", len(main_race_quali_2019))

Race rows: 420
Quali rows: 418


In [8]:
main_race_res_2019[['raceId','kaggle_driver_id']].duplicated().sum()
main_race_quali_2019[['raceId','kaggle_driver_id']].duplicated().sum()

np.int64(0)

In [9]:
train_2019 = main_race_res_2019.merge(
    main_race_quali_2019,
    on=["raceId","kaggle_driver_id"],
    how="left"
)

In [10]:
print("Rows before merge:", len(main_race_res_2019))
print("Rows after merge:", len(train_2019))

Rows before merge: 420
Rows after merge: 420


In [11]:
train_2019[['q1_time_sec','q2_time_sec','q3_time_sec']].isna().sum()

q1_time_sec     11
q2_time_sec    110
q3_time_sec    221
dtype: int64

In [12]:
print(train_2019.columns)

Index(['Position', 'Starting Grid', 'kaggle_driver_id', 'raceId', 'finished',
       'dnf', 'laps_down', 'time_gap_sec', 'year', 'Team', 'Track', 'rain',
       'sunny', 'position', 'participated_q2', 'participated_q3',
       'q1_time_sec', 'q2_time_sec', 'q3_time_sec'],
      dtype='object')


In [13]:
train_2019['participated_q3'].value_counts()

participated_q3
0.0    219
1.0    199
Name: count, dtype: int64

In [14]:
train_2019 = train_2019.rename(columns={
    "position": "quali_position"
})

In [66]:
def build_dataset(race_df, quali_df):
    df = race_df.merge(
        quali_df,
        on=["raceId", "kaggle_driver_id"],
        how="left"
    )

    # rename qualifying position
    if "position" in df.columns:
        df = df.rename(columns={"position": "quali_position"})

    # drop duplicate columns from quali table
    df = df.drop(columns=[
        c for c in ["Team_y", "Track_y", "rain_y", "sunny_y"] if c in df.columns
    ])

    # rename race columns
    # df = df.rename(columns={
    #     "Team_x": "Team",
    #     "Track_x": "Track",
    #     "rain_x": "rain",
    #     "sunny_x": "sunny"
    # })

    return df

In [77]:
train_2020 = build_dataset(main_race_res_2020, main_race_quali_2020)
train_2021 = build_dataset(main_race_res_2021, main_race_quali_2021)
train_2022 = build_dataset(main_race_res_2022, main_race_quali_2022)
train_2023 = build_dataset(main_race_res_2023, main_race_quali_2023)
train_2024 = build_dataset(main_race_res_2024, main_race_quali_2024)

In [68]:
print(len(main_race_res_2020), len(train_2020))
print(len(main_race_res_2021), len(train_2021))
print(len(main_race_res_2022), len(train_2022))
print(len(main_race_res_2023), len(train_2023))
print(len(main_race_res_2024), len(train_2024))

340 340
440 440
440 440
440 440
480 480


In [18]:
train_2024.groupby("raceId").size().value_counts()

20    24
Name: count, dtype: int64

In [78]:
train_df = pd.concat([
    train_2019,
    train_2020,
    train_2021,
    train_2022,
    train_2023,
    train_2024
], ignore_index=True)

In [84]:
train_df.to_csv("final_train_dataset_2019_2024.csv", index=False)

In [79]:
print(train_df.columns)

Index(['Position', 'Starting Grid', 'kaggle_driver_id', 'raceId', 'finished',
       'dnf', 'laps_down', 'time_gap_sec', 'year', 'Team', 'Track', 'rain',
       'sunny', 'quali_position', 'participated_q2', 'participated_q3',
       'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'Track_x'],
      dtype='object')


In [80]:
train_2024[['q1_time_sec','q2_time_sec','q3_time_sec']].isna().sum()

q1_time_sec      6
q2_time_sec    125
q3_time_sec    246
dtype: int64

In [83]:

train_df.drop(columns=['Track_x'], inplace=True)

In [24]:
train_df['Position'] = pd.to_numeric(train_df['Position'], errors='coerce').fillna(21).astype(int)

In [26]:
main_race_quali_2024['participated_q3'].value_counts()

participated_q3
0    240
1    238
Name: count, dtype: int64

In [27]:
train_df['participated_q3'].value_counts()

participated_q3
0.0    1058
1.0    1019
Name: count, dtype: int64

In [30]:
train_2024['participated_q3'].value_counts()

Series([], Name: count, dtype: int64)

In [32]:
train_df[train_df['year']==2024].head(10)

,Position,Starting Grid,kaggle_driver_id,raceId,finished,dnf,laps_down,time_gap_sec,year,Team,Track,rain,sunny,quali_position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec
2080,1,1.0,830,1121,1.0,0.0,0.0,0.000,2024,Red Bull Racing,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2081,2,5.0,815,1121,1.0,0.0,0.0,22.457,2024,Red Bull Racing,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2082,3,4.0,832,1121,1.0,0.0,0.0,25.110,2024,Ferrari,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2083,4,2.0,844,1121,1.0,0.0,0.0,39.669,2024,Ferrari,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2084,5,3.0,847,1121,1.0,0.0,0.0,46.788,2024,Mercedes,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2085,6,7.0,846,1121,1.0,0.0,0.0,48.458,2024,McLaren,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2086,7,9.0,1,1121,1.0,0.0,0.0,50.324,2024,Mercedes,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2087,8,8.0,857,1121,1.0,0.0,0.0,56.082,2024,McLaren,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2088,9,6.0,4,1121,1.0,0.0,0.0,74.887,2024,Aston Martin,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2089,10,12.0,840,1121,1.0,0.0,0.0,93.216,2024,Aston Martin,Bahrain,0,1,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
main_race_res_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Position          480 non-null    object 
 1   Starting Grid     479 non-null    float64
 2   kaggle_driver_id  480 non-null    int64  
 3   raceId            480 non-null    int64  
 4   finished          480 non-null    float64
 5   dnf               480 non-null    float64
 6   laps_down         426 non-null    float64
 7   time_gap_sec      288 non-null    float64
 8   year              480 non-null    int64  
 9   Team              480 non-null    object 
 10  Track             480 non-null    object 
 11  rain              480 non-null    int64  
 12  sunny             480 non-null    int64  
dtypes: float64(5), int64(5), object(3)
memory usage: 48.9+ KB


In [46]:
main_race_quali_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 478 entries, 0 to 477
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   raceId            478 non-null    int64  
 1   kaggle_driver_id  478 non-null    int64  
 2   position          478 non-null    object 
 3   participated_q2   478 non-null    int64  
 4   participated_q3   478 non-null    int64  
 5   q1_time_sec       474 non-null    float64
 6   q2_time_sec       355 non-null    float64
 7   q3_time_sec       234 non-null    float64
dtypes: float64(3), int64(4), object(1)
memory usage: 30.0+ KB


In [44]:
def check_raceid_alignment():
    datasets = {
        2019: (main_race_res_2019, main_race_quali_2019),
        2020: (main_race_res_2020, main_race_quali_2020),
        2021: (main_race_res_2021, main_race_quali_2021),
        2022: (main_race_res_2022, main_race_quali_2022),
        2023: (main_race_res_2023, main_race_quali_2023),
        2024: (main_race_res_2024, main_race_quali_2024),
    }

    for year, (race_df, quali_df) in datasets.items():
        race_ids = sorted(race_df['raceId'].unique().tolist())
        quali_ids = sorted(quali_df['raceId'].unique().tolist())

        print(f"\n=== {year} ===")
        print(f"Race  raceIds ({len(race_ids)}): {race_ids}")
        print(f"Quali raceIds ({len(quali_ids)}): {quali_ids}")

        if race_ids == quali_ids:
            print("✅ Perfect match")
        else:
            only_in_race = sorted(set(race_ids) - set(quali_ids))
            only_in_quali = sorted(set(quali_ids) - set(race_ids))
            print("❌ MISMATCH")
            if only_in_race:
                print(f"   In race NOT quali : {only_in_race}")
            if only_in_quali:
                print(f"   In quali NOT race : {only_in_quali}")

check_raceid_alignment()


=== 2019 ===
Race  raceIds (21): [1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030]
Quali raceIds (21): [1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030]
✅ Perfect match

=== 2020 ===
Race  raceIds (17): [1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047]
Quali raceIds (17): [1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047]
✅ Perfect match

=== 2021 ===
Race  raceIds (22): [1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1069, 1070, 1071, 1072, 1073]
Quali raceIds (22): [1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1069, 1070, 1071, 1072, 1073]
✅ Perfect match

=== 2022 ===
Race  raceIds (22): [1074, 1075, 1076, 1077, 1078,

In [74]:
def fix_raceid_by_track(race_df, quali_df):
    # Get ordered raceIds from race dataset (preserving original order, no sorting)
    race_raceids = list(dict.fromkeys(race_df['raceId'].tolist()))
    
    # Build a mapping: Track name -> raceId from race dataset
    track_to_raceid = {}
    for raceid in race_raceids:
        track = race_df[race_df['raceId'] == raceid]['Track'].iloc[0]
        track_to_raceid[track] = raceid
    
    print("Track -> raceId mapping from race dataset:")
    for track, rid in track_to_raceid.items():
        print(f"  {track}: {rid}")
    
    # Apply mapping to quali dataset
    quali_df = quali_df.copy()
    quali_df['raceId'] = quali_df['Track'].map(track_to_raceid)
    
    unmatched = quali_df['raceId'].isna().sum()
    if unmatched > 0:
        print(f"\n⚠️ {unmatched} rows in quali could not be matched (unknown Track names):")
        print(quali_df[quali_df['raceId'].isna()]['Track'].unique())
    else:
        print(f"\n✅ All quali raceIds remapped successfully")
    
    return quali_df

main_race_quali_2024 = fix_raceid_by_track(main_race_res_2024, main_race_quali_2024)

Track -> raceId mapping from race dataset:
  Bahrain: 1121
  Saudi Arabia: 1122
  Australia: 1123
  Japan: 1124
  China: 1125
  Miami: 1126
  Emilia Romagna: 1127
  Monaco: 1128
  Canada: 1129
  Spain: 1130
  Austria: 1131
  Great Britain: 1132
  Hungary: 1133
  Belgium: 1134
  Netherlands: 1135
  Italy: 1136
  Azerbaijan: 1137
  Singapore: 1138
  United States: 1139
  Mexico: 1140
  Brazil: 1141
  Las Vegas: 1142
  Qatar: 1143
  Abu Dhabi: 1144

✅ All quali raceIds remapped successfully


In [50]:
def check_raceid_alignment():
    datasets = {
        2024: (main_race_res_2024, main_race_quali_2024),
    }

    for year, (race_df, quali_df) in datasets.items():
        race_ids = sorted(race_df['raceId'].unique().tolist())
        quali_ids = sorted(quali_df['raceId'].unique().tolist())

        print(f"\n=== {year} ===")
        print(f"Race  raceIds ({len(race_ids)}): {race_ids}")
        print(f"Quali raceIds ({len(quali_ids)}): {quali_ids}")

        if race_ids == quali_ids:
            print("✅ Perfect match")
        else:
            only_in_race = sorted(set(race_ids) - set(quali_ids))
            only_in_quali = sorted(set(quali_ids) - set(race_ids))
            print("❌ MISMATCH")
            if only_in_race:
                print(f"   In race NOT quali : {only_in_race}")
            if only_in_quali:
                print(f"   In quali NOT race : {only_in_quali}")

check_raceid_alignment()


=== 2024 ===
Race  raceIds (24): [1121, 1122, 1123, 1124, 1125, 1126, 1127, 1128, 1129, 1130, 1131, 1132, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144]
Quali raceIds (24): [1121, 1122, 1123, 1124, 1125, 1126, 1127, 1128, 1129, 1130, 1131, 1132, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144]
✅ Perfect match


In [76]:
keep = ['raceId', 'kaggle_driver_id', 'position', 
            'participated_q2', 'participated_q3', 
            'q1_time_sec', 'q2_time_sec', 'q3_time_sec']
main_race_quali_2024 = main_race_quali_2024[keep]

In [58]:
for year, df in [('2019', train_2019), ('2020', train_2020), ('2021', train_2021), 
                  ('2022', train_2022), ('2023', train_2023), ('2024', train_2024)]:
    dirty_cols = [c for c in df.columns if '_x' in c or '_y' in c or 'Unnamed' in c]
    if dirty_cols:
        print(f"❌ {year}: {dirty_cols}")
    else:
        print(f"✅ {year}: clean")

✅ 2019: clean
❌ 2020: ['Team_x', 'Track_x', 'rain_x', 'sunny_x']
❌ 2021: ['Team_x', 'Track_x', 'rain_x', 'sunny_x']
❌ 2022: ['year_x', 'Team_x', 'Track_x', 'rain_x', 'sunny_x', 'year_y']
❌ 2023: ['year_x', 'Team_x', 'Track_x', 'rain_x', 'sunny_x', 'year_y']
❌ 2024: ['year_x', 'Team_x', 'Track_x', 'rain_x', 'sunny_x', 'year_y']


In [59]:
for year in [2019, 2020, 2021, 2022, 2023, 2024]:
    q = load_quali(year)
    print(f"{year}: {q.columns.tolist()}")

2019: ['raceId', 'Team', 'kaggle_driver_id', 'position', 'participated_q2', 'participated_q3', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'Track', 'rain', 'sunny']
2020: ['raceId', 'Team', 'kaggle_driver_id', 'position', 'participated_q2', 'participated_q3', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'Track', 'rain', 'sunny']
2021: ['raceId', 'Team', 'kaggle_driver_id', 'position', 'participated_q2', 'participated_q3', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'Track', 'rain', 'sunny']
2022: ['raceId', 'Track', 'kaggle_driver_id', 'position', 'participated_q2', 'participated_q3', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'year', 'Team', 'rain', 'sunny']
2023: ['raceId', 'Track', 'kaggle_driver_id', 'position', 'participated_q2', 'participated_q3', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'year', 'Team', 'rain', 'sunny']
2024: ['raceId', 'Track', 'kaggle_driver_id', 'position', 'participated_q2', 'participated_q3', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec', 'year', 'Team